In [4]:
"""
채용공고 지역 편중 지도 시각화 v3
- 지도 1: 시·도별 choropleth (전국)
- 지도 2: 서울 구별 choropleth
- 지도 3: 경기 시군별 choropleth  ← NEW
직무별 각 2장씩 총 6장 생성

※ 준비물 (GeoJSON):
   sido.geojson      - 시도 경계
   seoul_gu.geojson  - 서울 구 경계
   gyeonggi_si.geojson - 경기 시군 경계  ← NEW (seoul_gu.geojson과 동일 파일에서 필터)

   다운로드 (cmd):
   curl -L -o sido.geojson "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json"
   curl -L -o seoul_gu.geojson "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json"
   ※ gyeonggi_si.geojson은 seoul_gu.geojson(시군구 전체)에서 자동 필터링
"""

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# ────────────────────────────────────────────────
# 0. 경로 설정
# ────────────────────────────────────────────────
EXCEL_PATH   = r"C:\py_temp\중간프로젝트\posting_analysis_table_최종.xlsx"
SIDO_SHP     = r"C:\py_temp\중간프로젝트\sido.geojson"
SIGUNGU_SHP  = r"C:\py_temp\중간프로젝트\seoul_gu.geojson"   # 서울·경기 모두 이 파일에서 필터
OUTPUT_DIR   = r"C:\py_temp\중간프로젝트\output_maps_m3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ────────────────────────────────────────────────
# 1. 한글 폰트 설정
# ────────────────────────────────────────────────
def set_korean_font():
    candidates = [
        'C:/Windows/Fonts/malgun.ttf',
        'C:/Windows/Fonts/NanumGothic.ttf',
        '/System/Library/Fonts/AppleGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    ]
    for path in candidates:
        if os.path.exists(path):
            fe = fm.FontEntry(fname=path, name='KoreanFont')
            fm.fontManager.ttflist.insert(0, fe)
            plt.rcParams['font.family'] = 'KoreanFont'
            plt.rcParams['axes.unicode_minus'] = False
            print(f"✅ 폰트 설정: {path}")
            return
    for name in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']:
        try:
            fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
            plt.rcParams['font.family'] = name
            plt.rcParams['axes.unicode_minus'] = False
            print(f"✅ 폰트: {name}")
            return
        except Exception:
            pass
    print("⚠ 한글 폰트 미발견")

set_korean_font()

# ────────────────────────────────────────────────
# 2. 데이터 로드 & 전처리
# ────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH)

FOREIGN_KEYWORDS = ['미국', '일본', '베트남', '헝가리', '동경']
mask_foreign = df['시각화용_지역'].apply(
    lambda x: any(k in str(x) for k in FOREIGN_KEYWORDS)
)
df = df[~mask_foreign].copy()
print(f"국내 데이터: {len(df)}건")
print(df['job'].value_counts())

# ────────────────────────────────────────────────
# 3. 집계 함수
# ────────────────────────────────────────────────
def aggregate_sido(df, job_name):
    sub = df[df['job'] == job_name]
    total = len(sub)
    agg = sub.groupby('region_sido').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

def aggregate_seoul_gu(df, job_name):
    """서울 구별 - 인천 중구와 구분"""
    sub = df[(df['job'] == job_name) & (df['region_sido'] == '서울')].copy()
    total = len(sub)
    sub['gu'] = sub['시각화용_지역'].str.replace('^서울 ', '', regex=True).str.strip()
    sub.loc[sub['gu'] == '서울', 'gu'] = '기타'
    agg = sub.groupby('gu').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

def aggregate_gyeonggi_si(df, job_name):
    """경기 시군별"""
    sub = df[(df['job'] == job_name) & (df['region_sido'] == '경기')].copy()
    total = len(sub)
    sub['si'] = sub['시각화용_지역'].str.replace('^경기 ', '', regex=True).str.strip()
    agg = sub.groupby('si').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

def aggregate_seoul_gyeonggi(df, job_name):
    """서울 구별 + 경기 시군별 통합 집계"""
    # 서울
    sub_s = df[(df['job'] == job_name) & (df['region_sido'] == '서울')].copy()
    sub_s['area'] = sub_s['시각화용_지역'].str.replace('^서울 ', '', regex=True).str.strip()
    sub_s.loc[sub_s['area'] == '서울', 'area'] = '기타'

    # 경기
    sub_g = df[(df['job'] == job_name) & (df['region_sido'] == '경기')].copy()
    sub_g['area'] = sub_g['시각화용_지역'].str.replace('^경기 ', '', regex=True).str.strip()

    combined = pd.concat([sub_s[['area']], sub_g[['area']]])
    total = len(combined)
    agg = combined.groupby('area').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total
# ────────────────────────────────────────────────
# 4. 색상 팔레트
# ────────────────────────────────────────────────
CMAP_DA = LinearSegmentedColormap.from_list(
    'da_blue', ['#EBF5FB', '#AED6F1', '#5DADE2', '#2874A6', '#1A5276'], N=256)
CMAP_BE = LinearSegmentedColormap.from_list(
    'be_green', ['#E9F7EF', '#A9DFBF', '#52BE80', '#1E8449', '#145A32'], N=256)

JOB_CONFIG = {
    '데이터 분석가': {'cmap': CMAP_DA, 'accent': '#2874A6', 'prefix': 'DA'},
    '백엔드 개발자':  {'cmap': CMAP_BE, 'accent': '#1E8449', 'prefix': 'BE'},
}

# ────────────────────────────────────────────────
# 5. 시도 이름 매핑
# ────────────────────────────────────────────────
SIDO_NAME_MAP = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원특별자치도': '강원', '강원도': '강원',
    '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북',
    '전라남도': '전남', '경상북도': '경북', '경상남도': '경남',
    '제주특별자치도': '제주',
    'Seoul': '서울', 'Busan': '부산', 'Daegu': '대구',
    'Incheon': '인천', 'Gwangju': '광주', 'Daejeon': '대전',
    'Ulsan': '울산', 'Sejong': '세종',
    'Gyeonggi-do': '경기', 'Gangwon-do': '강원',
    'Chungcheongbuk-do': '충북', 'Chungcheongnam-do': '충남',
    'Jeollabuk-do': '전북', 'Jeollanam-do': '전남',
    'Gyeongsangbuk-do': '경북', 'Gyeongsangnam-do': '경남',
    'Jeju-do': '제주',
}

# ────────────────────────────────────────────────
# 6. 레이블 오프셋 (겹침 방지)
# ────────────────────────────────────────────────
SIDO_LABEL_OFFSET = {
    '경기': (0.45, 0.10),
    '서울': (-0.05, 0.10),
    '인천': (-0.30, -0.20),
    '세종': (-0.35, 0.10),
}

SEOUL_GU_LABEL_OFFSET = {
    '중구':   (0.005, 0.005),
    '용산구': (0.0,  -0.005),
    '종로구': (0.0,   0.005),
    '양천구': (-0.01, 0.0),
}

# 경기 시군 오프셋 (필요시 추가)
GYEONGGI_SI_LABEL_OFFSET = {
    '성남시': (0.0,   0.02),
    '과천시': (-0.02, 0.0),
    '의왕시': (0.02,  0.0),
    '안양시': (-0.03, 0.0),
    '군포시': (0.0,  -0.02),
}

# ────────────────────────────────────────────────
# 7. 공통 유틸
# ────────────────────────────────────────────────
def make_stroke(color='white', linewidth=2.5):
    return [pe.withStroke(linewidth=linewidth, foreground=color)]

def text_color_and_stroke(norm_val, threshold=0.45):
    if norm_val > threshold:
        return 'white', make_stroke('black', 2.0)
    else:
        return '#1A1A1A', make_stroke('white', 2.5)

def find_name_col(gdf, candidates):
    return next((c for c in candidates if c in gdf.columns), None)

def load_geo(path):
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

# ────────────────────────────────────────────────
# 8. 시도별 지도
# ────────────────────────────────────────────────
def draw_sido_map(job_name, sido_gdf_raw, df, output_dir):
    cfg = JOB_CONFIG[job_name]
    agg, total = aggregate_sido(df, job_name)

    name_col = find_name_col(sido_gdf_raw, ['name','CTP_KOR_NM','SIDO_NM','kor_name','NAME_1'])
    if not name_col:
        print("⚠ 시도 이름 컬럼 없음"); return

    gdf = sido_gdf_raw.copy()
    gdf['sido_std'] = gdf[name_col].map(SIDO_NAME_MAP).fillna(gdf[name_col])
    gdf = gdf.merge(agg, left_on='sido_std', right_on='region_sido', how='left')
    gdf['count'] = gdf['count'].fillna(0)
    gdf['pct']   = gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(13, 15), facecolor='#F5F5F5')
    ax.set_facecolor('#F0F4F8')
    vmax = gdf['count'].max()

    gdf.plot(column='count', cmap=cfg['cmap'], linewidth=0.7,
             edgecolor='#777777', ax=ax, vmin=0, vmax=vmax,
             missing_kwds={'color': '#DDDDDD'})

    for _, row in gdf.iterrows():
        if row['count'] == 0:
            continue
        c = row.geometry.centroid
        x, y = c.x, c.y
        dx, dy = SIDO_LABEL_OFFSET.get(row['sido_std'], (0, 0))
        x += dx; y += dy
        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val)
        ax.annotate(
            f"{row['sido_std']}\n{int(row['count'])}건  ({row['pct']}%)",
            xy=(x, y), ha='center', va='center',
            fontsize=8.5, fontweight='bold', color=fc,
            path_effects=stroke, linespacing=1.5,
        )

    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    job_label = '데이터 분석가' if job_name == '데이터 분석가' else '백엔드 개발자'
    ax.set_title(f"{job_label} 채용공고 지역 분포\n시·도별  |  총 {total}건",
                 fontsize=15, fontweight='bold', pad=16, color='#1A1A1A', linespacing=1.8)
    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_sido_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 9. 서울 구별 지도
# ────────────────────────────────────────────────
def draw_seoul_map(job_name, seoul_gdf_raw, df, output_dir):
    cfg = JOB_CONFIG[job_name]
    agg, total_seoul = aggregate_seoul_gu(df, job_name)

    name_col = find_name_col(seoul_gdf_raw, ['name','SIG_KOR_NM','GU_NM','kor_name','NAME_2','sggnm'])
    if not name_col:
        print("⚠ 구 이름 컬럼 없음"); return

    gdf = seoul_gdf_raw.copy()
    gdf['gu_std'] = gdf[name_col].str.strip()
    gdf = gdf.merge(agg, left_on='gu_std', right_on='gu', how='left')
    gdf['count'] = gdf['count'].fillna(0)
    gdf['pct']   = gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(15, 13), facecolor='#F5F5F5')
    ax.set_facecolor('#EFF4F9')
    vmax = gdf['count'].max()

    gdf.plot(column='count', cmap=cfg['cmap'], linewidth=0.9,
             edgecolor='#666666', ax=ax, vmin=0, vmax=vmax,
             missing_kwds={'color': '#DDDDDD'})

    for _, row in gdf.iterrows():
        c = row.geometry.centroid
        x, y = c.x, c.y
        gu = row['gu_std']
        dx, dy = SEOUL_GU_LABEL_OFFSET.get(gu, (0, 0))
        x += dx; y += dy
        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val, threshold=0.38)

        if row['count'] > 0:
            label = f"{gu}\n{int(row['count'])}건  ({row['pct']}%)"
            fs = 8.0
        else:
            label = gu
            fs = 7.0
            fc = '#888888'
            stroke = make_stroke('white', 2.0)

        ax.annotate(label, xy=(x, y), ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=fc,
                    path_effects=stroke, linespacing=1.5)

    # 강남구 강조 테두리
    gangnam = gdf[gdf['gu_std'] == '강남구']
    if not gangnam.empty:
        gangnam.boundary.plot(ax=ax, linewidth=3.0, edgecolor='#E74C3C', zorder=5)

    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    job_label = '데이터 분석가' if job_name == '데이터 분석가' else '백엔드 개발자'
    ax.set_title(f"{job_label} 채용공고 지역 분포\n서울시 구별  |  서울 내 총 {total_seoul}건",
                 fontsize=15, fontweight='bold', pad=16, color='#1A1A1A', linespacing=1.8)
    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_seoul_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 10. 경기 시군별 지도  ← NEW
# ────────────────────────────────────────────────
def draw_gyeonggi_map(job_name, gg_gdf_raw, df, output_dir):
    cfg = JOB_CONFIG[job_name]
    agg, total_gg = aggregate_gyeonggi_si(df, job_name)

    name_col = find_name_col(gg_gdf_raw, ['name','SIG_KOR_NM','GU_NM','kor_name','NAME_2','sggnm'])
    if not name_col:
        print("⚠ 시군 이름 컬럼 없음"); return

    gdf = gg_gdf_raw.copy()
    gdf['si_std'] = gdf[name_col].str.strip()
    gdf = gdf.merge(agg, left_on='si_std', right_on='si', how='left')
    gdf['count'] = gdf['count'].fillna(0)
    gdf['pct']   = gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(14, 13), facecolor='#F5F5F5')
    ax.set_facecolor('#EFF4F9')
    vmax = gdf['count'].max()

    gdf.plot(column='count', cmap=cfg['cmap'], linewidth=0.9,
             edgecolor='#666666', ax=ax, vmin=0, vmax=vmax,
             missing_kwds={'color': '#DDDDDD'})

    for _, row in gdf.iterrows():
        c = row.geometry.centroid
        x, y = c.x, c.y
        si = row['si_std']
        dx, dy = GYEONGGI_SI_LABEL_OFFSET.get(si, (0, 0))
        x += dx; y += dy
        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val, threshold=0.40)

        if row['count'] > 0:
            label = f"{si}\n{int(row['count'])}건  ({row['pct']}%)"
            fs = 7.5
        else:
            label = si
            fs = 6.5
            fc = '#999999'
            stroke = make_stroke('white', 2.0)

        ax.annotate(label, xy=(x, y), ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=fc,
                    path_effects=stroke, linespacing=1.5)

    # 성남시 강조 테두리 (판교 = 경기의 강남구)
    seongnam = gdf[gdf['si_std'] == '성남시']
    if not seongnam.empty:
        seongnam.boundary.plot(ax=ax, linewidth=3.0, edgecolor='#E74C3C', zorder=5)

    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    job_label = '데이터 분석가' if job_name == '데이터 분석가' else '백엔드 개발자'
    ax.set_title(f"{job_label} 채용공고 지역 분포\n경기도 시군별  |  경기 내 총 {total_gg}건",
                 fontsize=15, fontweight='bold', pad=16, color='#1A1A1A', linespacing=1.8)
    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_gyeonggi_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")


def draw_seoul_gyeonggi_map(job_name, seoul_gdf_raw, gg_gdf_raw, df, output_dir):
    """서울 구별 + 경기 시군별 통합 지도"""
    cfg = JOB_CONFIG[job_name]
    agg, total = aggregate_seoul_gyeonggi(df, job_name)

    # 서울 GDF 준비
    name_col_s = find_name_col(seoul_gdf_raw, ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2'])
    gdf_s = seoul_gdf_raw.copy()
    gdf_s['area_std'] = gdf_s[name_col_s].str.strip()

    # 경기 GDF 준비
    name_col_g = find_name_col(gg_gdf_raw, ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2'])
    gdf_g = gg_gdf_raw.copy()
    gdf_g['area_std'] = gdf_g[name_col_g].str.strip()

    # 합치기
    gdf = pd.concat([gdf_s[['area_std','geometry']], gdf_g[['area_std','geometry']]],
                    ignore_index=True)
    gdf = gpd.GeoDataFrame(gdf, crs="EPSG:4326")
    gdf = gdf.merge(agg, left_on='area_std', right_on='area', how='left')
    gdf['count'] = gdf['count'].fillna(0)
    gdf['pct']   = gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(15, 14), facecolor='#F5F5F5')
    ax.set_facecolor('#EFF4F9')
    vmax = gdf['count'].max()

    gdf.plot(column='count', cmap=cfg['cmap'], linewidth=0.9,
             edgecolor='#666666', ax=ax, vmin=0, vmax=vmax,
             missing_kwds={'color': '#DDDDDD'})

    for _, row in gdf.iterrows():
        c = row.geometry.centroid
        x, y = c.x, c.y
        area = row['area_std']
        # 오프셋 적용 (서울 구 + 경기 시 통합)
        offset_map = {**SEOUL_GU_LABEL_OFFSET, **GYEONGGI_SI_LABEL_OFFSET}
        dx, dy = offset_map.get(area, (0, 0))
        x += dx; y += dy
        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val, threshold=0.40)

        if row['count'] > 0:
            label = f"{area}\n{int(row['count'])}건  ({row['pct']}%)"
            fs = 7.0
        else:
            label = area
            fs = 6.0
            fc = '#999999'
            stroke = make_stroke('white', 2.0)

        ax.annotate(label, xy=(x, y), ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=fc,
                    path_effects=stroke, linespacing=1.5)

    # 강남구 강조 테두리
    gangnam = gdf[gdf['area_std'] == '강남구']
    if not gangnam.empty:
        gangnam.boundary.plot(ax=ax, linewidth=3.0, edgecolor='#E74C3C', zorder=5)

    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    job_label = '데이터 분석가' if job_name == '데이터 분석가' else '백엔드 개발자'
    ax.set_title(f"{job_label} 채용공고 지역 분포\n서울·경기 시군구별  |  서울+경기 총 {total}건",
                 fontsize=15, fontweight='bold', pad=16, color='#1A1A1A', linespacing=1.8)
    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_seoul_gyeonggi_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")

def aggregate_seoul_gyeonggi(df, job_name):
    """서울 구별 + 경기 시별 통합 집계 (서울+경기 합산 기준 %)"""
    sub_s = df[(df['job'] == job_name) & (df['region_sido'] == '서울')].copy()
    sub_s['area'] = sub_s['시각화용_지역'].str.replace('^서울 ', '', regex=True).str.strip()
    sub_s.loc[sub_s['area'] == '서울', 'area'] = '기타'

    sub_g = df[(df['job'] == job_name) & (df['region_sido'] == '경기')].copy()
    sub_g['area'] = sub_g['시각화용_지역'].str.replace('^경기 ', '', regex=True).str.strip()

    combined = pd.concat([sub_s[['area']], sub_g[['area']]])
    total = len(combined)
    agg = combined.groupby('area').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total  

def dissolve_gyeonggi_to_city(gg_gdf_raw, name_col):
    """
    경기 GeoJSON은 대도시(성남시·수원시·고양시·용인시·안양시 등)를
    구 단위로 쪼개 저장 → 시 단위로 dissolve 해야 데이터와 매칭됨.
    예) '성남시 분당구', '성남시 수정구' → '성남시' 로 병합
    """
    gdf = gg_gdf_raw.copy()
    # 이름에서 시·군 앞부분 추출: "성남시 분당구" → "성남시", "과천시" → "과천시"
    gdf['city'] = gdf[name_col].str.strip().str.extract(r'^(\S+[시군])')[0]
    gdf['city'] = gdf['city'].fillna(gdf[name_col].str.strip())
    dissolved = gdf.dissolve(by='city', as_index=False)[['city', 'geometry']]
    dissolved = dissolved.rename(columns={'city': 'area_std'})
    return dissolved

def draw_seoul_gyeonggi_map(job_name, seoul_gdf_raw, gg_gdf_raw, df, output_dir):
    """서울 구별 + 경기 시별 통합 지도"""
    cfg = JOB_CONFIG[job_name]
    agg, total = aggregate_seoul_gyeonggi(df, job_name)

    # 서울 GDF
    name_col_s = find_name_col(seoul_gdf_raw, ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2'])
    gdf_s = seoul_gdf_raw.copy()
    gdf_s = gdf_s[[name_col_s, 'geometry']].rename(columns={name_col_s: 'area_std'})
    gdf_s['area_std'] = gdf_s['area_std'].str.strip()

    # 경기 GDF — 시 단위로 dissolve (핵심 수정)
    name_col_g = find_name_col(gg_gdf_raw, ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2'])
    gdf_g = dissolve_gyeonggi_to_city(gg_gdf_raw, name_col_g)

    # 합치기
    gdf = pd.concat([gdf_s, gdf_g], ignore_index=True)
    gdf = gpd.GeoDataFrame(gdf, crs="EPSG:4326")
    gdf = gdf.merge(agg, left_on='area_std', right_on='area', how='left')
    gdf['count'] = gdf['count'].fillna(0)
    gdf['pct']   = gdf['pct'].fillna(0)

    fig, ax = plt.subplots(figsize=(15, 14), facecolor='#F5F5F5')
    ax.set_facecolor('#EFF4F9')
    vmax = gdf['count'].max()

    gdf.plot(column='count', cmap=cfg['cmap'], linewidth=0.9,
             edgecolor='#666666', ax=ax, vmin=0, vmax=vmax,
             missing_kwds={'color': '#DDDDDD'})

    for _, row in gdf.iterrows():
        c = row.geometry.centroid
        x, y = c.x, c.y
        area = row['area_std']
        offset_map = {**SEOUL_GU_LABEL_OFFSET, **GYEONGGI_SI_LABEL_OFFSET}
        dx, dy = offset_map.get(area, (0, 0))
        x += dx; y += dy
        norm_val = row['count'] / vmax if vmax > 0 else 0
        fc, stroke = text_color_and_stroke(norm_val, threshold=0.40)

        if row['count'] > 0:
            label = f"{area}\n{int(row['count'])}건  ({row['pct']}%)"
            fs = 7.0
        else:
            label = area
            fs = 6.0
            fc = '#999999'
            stroke = make_stroke('white', 2.0)

        ax.annotate(label, xy=(x, y), ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=fc,
                    path_effects=stroke, linespacing=1.5)

    # 강남구 강조 테두리
    gangnam = gdf[gdf['area_std'] == '강남구']
    if not gangnam.empty:
        gangnam.boundary.plot(ax=ax, linewidth=3.0, edgecolor='#E74C3C', zorder=5)

    sm = plt.cm.ScalarMappable(cmap=cfg['cmap'],
                                norm=mcolors.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.022, pad=0.01, shrink=0.55, aspect=25)
    cbar.ax.tick_params(labelsize=9)
    cbar.ax.set_ylabel('공고 건수', fontsize=10, rotation=270, labelpad=15)

    job_label = '데이터 분석가' if job_name == '데이터 분석가' else '백엔드 개발자'
    seoul_total = len(df[(df['job'] == job_name) & (df['region_sido'] == '서울')])
    gg_total    = len(df[(df['job'] == job_name) & (df['region_sido'] == '경기')])
    ax.set_title(
        f"{job_label} 채용공고 지역 분포\n"
        f"서울·경기  |  서울 {seoul_total}건 + 경기 {gg_total}건 = 총 {total}건",
        fontsize=15, fontweight='bold', pad=16, color='#1A1A1A', linespacing=1.8
    )
    ax.axis('off')
    plt.tight_layout(pad=1.5)

    fpath = os.path.join(output_dir, f"{cfg['prefix']}_seoul_gyeonggi_map.png")
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#F5F5F5')
    plt.close()
    print(f"저장: {fpath}")  
# ────────────────────────────────────────────────
# 11. 실행
# ────────────────────────────────────────────────
SEOUL_GU_LIST = [
    '강남구','강동구','강북구','강서구','관악구','광진구','구로구','금천구',
    '노원구','도봉구','동대문구','동작구','마포구','서대문구','서초구',
    '성동구','성북구','송파구','양천구','영등포구','용산구','은평구',
    '종로구','중구','중랑구'
]

# 경기 시군 목록 (필터용)
GYEONGGI_SI_LIST = [
    '수원시','성남시','의정부시','안양시','부천시','광명시','평택시','동두천시',
    '안산시','고양시','과천시','구리시','남양주시','오산시','시흥시','군포시',
    '의왕시','하남시','용인시','파주시','이천시','안성시','김포시','화성시',
    '광주시','양주시','포천시','여주시','연천군','가평군','양평군',
]

def filter_sigungu(gdf, sido_code_prefix, name_list, name_col):
    """시군구 전체 GeoJSON에서 특정 시도만 필터"""
    if 'CTPRVN_CD' in gdf.columns:
        return gdf[gdf['CTPRVN_CD'] == sido_code_prefix].copy()
    if 'code' in gdf.columns:
        return gdf[gdf['code'].astype(str).str.startswith(sido_code_prefix)].copy()
    # 이름으로 필터 (fallback)
    return gdf[gdf[name_col].isin(name_list)].copy()

def main():
    for path, label in [(SIDO_SHP, '시도'), (SIGUNGU_SHP, '시군구')]:
        if not os.path.exists(path):
            print(f"[ERROR] {label} 파일 없음: {path}")
            return

    print("시도 로드...")
    sido_gdf = load_geo(SIDO_SHP)
    print(f"  컬럼: {sido_gdf.columns.tolist()}")

    print("시군구 로드...")
    sigungu_all = load_geo(SIGUNGU_SHP)
    print(f"  컬럼: {sigungu_all.columns.tolist()}")

    name_col = find_name_col(sigungu_all, ['name','SIG_KOR_NM','GU_NM','sggnm','NAME_2'])

    # 서울 구 필터 (코드: 11)
    seoul_gu  = filter_sigungu(sigungu_all, '11', SEOUL_GU_LIST, name_col)
    print(f"  서울 구 수: {len(seoul_gu)}")

    # 경기 시군 필터 - 이름 기반으로 직접 필터
    gyeonggi_si = sigungu_all[sigungu_all[name_col].isin(GYEONGGI_SI_LIST)].copy()
    print(f"  경기 시군 수: {len(gyeonggi_si)}")

    # 필터 결과 확인
    if len(gyeonggi_si) == 0:
        print("⚠ 경기 시군 필터 실패. 컬럼 값 샘플:")
        print(sigungu_all[name_col].head(20).tolist())

    for job in ['데이터 분석가', '백엔드 개발자']:
        print(f"\n{'='*40}\n{job} 지도 생성\n{'='*40}")
        draw_sido_map(job, sido_gdf, df, OUTPUT_DIR)
        draw_seoul_gyeonggi_map(job, seoul_gu, gyeonggi_si, df, OUTPUT_DIR)

    print(f"\n✅ 완료! → {os.path.abspath(OUTPUT_DIR)}/")
    print("생성 파일:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"  - {f}")

if __name__ == "__main__":
    main()


✅ 폰트 설정: C:/Windows/Fonts/malgun.ttf
국내 데이터: 877건
job
백엔드 개발자    603
데이터 분석가    274
Name: count, dtype: int64
시도 로드...
  컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
시군구 로드...
  컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
  서울 구 수: 25
  경기 시군 수: 25

데이터 분석가 지도 생성
저장: C:\py_temp\중간프로젝트\output_maps_m3\DA_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m3\DA_seoul_gyeonggi_map.png

백엔드 개발자 지도 생성
저장: C:\py_temp\중간프로젝트\output_maps_m3\BE_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m3\BE_seoul_gyeonggi_map.png

✅ 완료! → C:\py_temp\중간프로젝트\output_maps_m3/
생성 파일:
  - BE_gyeonggi_map.png
  - BE_seoul_gyeonggi_map.png
  - BE_seoul_map.png
  - BE_sido_map.png
  - DA_gyeonggi_map.png
  - DA_seoul_gyeonggi_map.png
  - DA_seoul_map.png
  - DA_sido_map.png
